In [5]:
import pandas as pd

In [ ]:
# df = pd.read_parquet('../data/sessions/274d8b85/df.parquet')
df

In [8]:
import os 

os.getcwd()

'/home/coder/dev/dynamic_pricing/backend'

In [6]:
from src.download_data import download_data
download_data(["inflacion", "transacciones"])

Download complete.
Download complete.


In [1]:
from src.get_data import GetData
from api.session_manager import SessionManager
session_manager = SessionManager()
# session_id = session_manager.create_session()

# data_loader = GetData(session_id=session_id)
# df, features = data_loader.get_data()

In [ ]:
from fastapi import HTTPException

# Guardar datos en la sesión
session_manager.save_dataframe(session_id, df, "df")
session_manager.save_object(session_id, list(features), "features")

# Calcular stats
metadata = {
    "session_id": session_id,
    "rows": len(df),
    "products": df['ProductID'].nunique(),
    "customers": df['CustomerID'].nunique(),
    "date_range": {
        "start": str(df['Date'].min()),
        "end": str(df['Date'].max())
    },
    "features": list(features)
}

session_manager.save_metadata(session_id, metadata)
if not session_manager.session_exists(session_id):
        raise HTTPException(status_code=404, detail=f"Session {session_id} not found")
    
metadata = session_manager.load_metadata(session_id)
metadata
    

In [2]:
session_id = '685ea378'

df = session_manager.load_dataframe(session_id, "df")

In [20]:
df[(df['ProductID'] == 'Books_B')][['Price']].mean()

Price    54.973131
dtype: float64

In [16]:
df[(df['ProductID'] == 'Books_A')][['Price','demand']].sort_values(['demand']).head(5000)

,Price,demand
1562,99.134031,1
97467,90.622628,1
97533,91.907022,1
31058,95.125373,1
30538,94.160623,1
...,...,...
8483,20.507909,6
58001,21.907430,6
52200,14.264577,6
51844,30.085976,6


In [ ]:
from api.routes import data, model, analysis, recommendations
import asyncio

train_response = await model.train_model('685ea378')

In [5]:
from api.routes import analysis
from api.models.schemas import ElasticityRequest, OptimizeRequest, ScenariosRequest
import asyncio
product_id = 'Books_B'
request = ElasticityRequest(product_id=product_id)
response = await analysis.get_elasticity(session_id, request)

print(f"\n{'='*60}")
print(f"ANÁLISIS DE ELASTICIDAD - {product_id}")
print(f"{'='*60}")
print(f"📊 Elasticidad: {response.elasticity:.4f}")
print(f"📈 Tipo: {response.elasticity_type}")
print(f"💰 Precio actual: ${response.current_price:.2f}")
print(f"\n📉 Datos para gráfico:")
print(f"   Rango de precios: ${response.prices[0]:.2f} - ${response.prices[-1]:.2f}")
print(f"   Demanda estimada: {response.demands[0]:.0f} - {response.demands[-1]:.0f} unidades")



ANÁLISIS DE ELASTICIDAD - Books_B
📊 Elasticidad: -0.1568
📈 Tipo: inelastic
💰 Precio actual: $50.00

📉 Datos para gráfico:
   Rango de precios: $35.00 - $65.00
   Demanda estimada: 3 - 2 unidades


In [7]:
from api.routes import analysis
from api.models.schemas import ElasticityRequest, OptimizeRequest, ScenariosRequest
import asyncio
product_id = 'Home Decor_B'
request = ElasticityRequest(product_id=product_id)
response = await analysis.get_elasticity(session_id, request)

print(f"\n{'='*60}")
print(f"ANÁLISIS DE ELASTICIDAD - {product_id}")
print(f"{'='*60}")
print(f"📊 Elasticidad: {response.elasticity:.4f}")
print(f"📈 Tipo: {response.elasticity_type}")
print(f"💰 Precio actual: ${response.current_price:.2f}")
print(f"\n📉 Datos para gráfico:")
print(f"   Rango de precios: ${response.prices[0]:.2f} - ${response.prices[-1]:.2f}")
print(f"   Demanda estimada: {response.demands[0]:.0f} - {response.demands[-1]:.0f} unidades")



ANÁLISIS DE ELASTICIDAD - Home Decor_B
📊 Elasticidad: -1.3612
📈 Tipo: elastic
💰 Precio actual: $50.00

📉 Datos para gráfico:
   Rango de precios: $35.00 - $65.00
   Demanda estimada: 5 - 3 unidades
